# 🌍 Evrensel Biyolojik Turnuva (5 Farklı Coğrafya)

Bu notebook, 18 farklı Sinek Mimarisini ve PSO'yu sadece Rastrigin'de değil, **literatürdeki en ünlü 5 optimizasyon cehenneminde** yarıştırarak biyolojik beynin hangi tür coğrafyalarda (vadiler, düzlükler, sivri tepeler) daha üstün olduğunu keşfeder.

**Savaş Alanları:**
1. **Sphere:** Basit düzlük.
2. **Rastrigin:** Bol tuzaklı, simetrik çukurlar.
3. **Ackley:** Uçsuz bucaksız düzlük ve ortasında devasa, dar bir kara delik.
4. **Rosenbrock:** (Banana Valley) İnce, kavisli, uzun ve ölümcül bir kanyon.
5. **Beale:** Asimetrik uçurumlar ve garip çukurlar.

In [ ]:
# 1. GEREKSİNİMLER VE AĞ YÜKLEMESİ
from google.colab import drive
import sys, os, glob, time
import torch
import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

drive.mount('/content/drive')
project_path = '/content/drive/MyDrive/fly_op'
if not os.path.exists(project_path): project_path = '/content/drive/MyDrive/fly_op/fly_op'
sys.path.append(project_path); sys.path.append(os.path.join(project_path, 'src')); os.chdir(project_path)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from flyopt.variants.rate_brain import RateBrain, RateBrainConfig, build_subgraph_bfs, select_connected_encode_decode

# --- 5 FARKLI ARAZİ (BENCHMARK) ---
def sphere(x): return torch.sum(x**2, dim=1)

def rastrigin(x): return 10 * 2 + torch.sum(x**2 - 10 * torch.cos(2 * np.pi * x), dim=1)

def ackley(x):
    sum_sq = torch.sum(x**2, dim=1)
    sum_cos = torch.sum(torch.cos(2 * np.pi * x), dim=1)
    return -20 * torch.exp(-0.2 * torch.sqrt(0.5 * sum_sq)) - torch.exp(0.5 * sum_cos) + 20 + np.e

def rosenbrock(x):
    return (1 - x[:,0])**2 + 100 * (x[:,1] - x[:,0]**2)**2

def beale(x):
    return (1.5 - x[:,0] + x[:,0]*x[:,1])**2 + (2.25 - x[:,0] + x[:,0]*(x[:,1]**2))**2 + (2.625 - x[:,0] + x[:,0]*(x[:,1]**3))**2

terrains = {
    "Sphere": sphere,
    "Rastrigin": rastrigin,
    "Ackley": ackley,
    "Rosenbrock": rosenbrock,
    "Beale": beale
}

# AĞ YÜKLEME
found_files = glob.glob('/content/drive/MyDrive/**/malecns_adjacency.npz', recursive=True)
DATA_PROCESSED = os.path.dirname(found_files[0]) if found_files else f"{project_path}/data/processed"
base_weights = sparse.load_npz(f"{DATA_PROCESSED}/malecns_adjacency.npz")
afferent, efferent = np.load(f"{DATA_PROCESSED}/malecns_afferent_indices.npy"), np.load(f"{DATA_PROCESSED}/malecns_efferent_indices.npy")

encode_full, decode_full = select_connected_encode_decode(base_weights, afferent, efferent, n_encode=4, n_decode=4, max_hops=6, n_encode_candidates=200, seed=9000)
sub_real, encode_idx, decode_idx, _ = build_subgraph_bfs(base_weights, encode_full, decode_full, 3000, seed=9000)
cfg = RateBrainConfig(dim=2, n_readout=len(decode_idx), T=20, decode_scale=0.5, train_gain=True)
brain = RateBrain(sub_real, encode_idx, decode_idx, cfg, seed=42).to(device)
print("✅ Beyin Yüklendi! Evrensel Turnuvaya Hazır.")

In [ ]:
# 2. ALGORİTMALAR
def pso_optimize(func, num_particles=1000, iters=100):
    x = (torch.rand((num_particles, 2), device=device) * 10 - 5)
    v = torch.zeros_like(x)
    pbest, pbest_obj = x.clone(), func(x)
    gbest_obj = torch.min(pbest_obj)
    gbest = pbest[torch.argmin(pbest_obj)].clone()
    for _ in range(iters):
        r1, r2 = torch.rand((num_particles, 2), device=device), torch.rand((num_particles, 2), device=device)
        v = 0.5 * v + 1.5 * r1 * (pbest - x) + 1.5 * r2 * (gbest - x)
        x = x + v
        obj = func(x)
        mask = obj < pbest_obj
        pbest[mask], pbest_obj[mask] = x[mask], obj[mask]
        if torch.min(pbest_obj) < gbest_obj:
            gbest_obj = torch.min(pbest_obj)
            gbest = pbest[torch.argmin(pbest_obj)].clone()
    return gbest_obj.item()

def multi_variant_fly(func, koku_tipi, suru_tipi, enerji_tipi, num_particles=1000, iters=100):
    x = (torch.rand((num_particles, 2), device=device) * 10 - 5)
    x.requires_grad_(True)
    best_obj = func(x).detach()
    gbest_obj = torch.min(best_obj)
    gbest_x = x[torch.argmin(best_obj)].detach().clone()
    momentum = torch.zeros_like(x, device=device)
    
    for i in range(iters):
        obj = func(x)
        obj.sum().backward()
        with torch.no_grad():
            grad = x.grad.clone()
            x.grad.zero_()
            
            if koku_tipi == 'Kör': vec12 = x.detach() * 0.1 
            elif koku_tipi == 'Anlık': vec12 = torch.nn.functional.normalize(-grad, p=2, dim=1)
            elif koku_tipi == 'Hafızalı':
                momentum = 0.8 * momentum + 0.2 * grad
                vec12 = torch.nn.functional.normalize(-momentum, p=2, dim=1)
                
            swarm_vec = torch.nn.functional.normalize(gbest_x - x.detach(), p=2, dim=1)
            swarm_vec = torch.nan_to_num(swarm_vec, 0.0)
            
            if suru_tipi == 'Yok': vec34 = torch.zeros_like(vec12)
            elif suru_tipi == 'Yan_Lob': vec34 = swarm_vec
            elif suru_tipi == 'Ana_Lob': vec12, vec34 = swarm_vec, torch.zeros_like(vec12)
                
            env_input = torch.cat([vec12, vec34], dim=1)
            fly_step = brain(env_input)
            
            lr = 0.1 if enerji_tipi == 'Sabit' else max(0.005, 0.2 * (1.0 - (i / iters)))
            x_new = x.detach() + fly_step[:, :2] * lr
            
            new_obj = func(x_new)
            mask = new_obj < best_obj
            x_updated = x.detach().clone()
            x_updated[mask] = x_new[mask]
            x = x_updated.clone()
            x.requires_grad_(True)
            best_obj[mask] = new_obj[mask]
            
            if torch.min(best_obj) < gbest_obj:
                gbest_obj = torch.min(best_obj)
                gbest_x = x[torch.argmin(best_obj)].detach().clone()
    return gbest_obj.item()

In [ ]:
# 3. TÜM ARAZİLERDE TURNUVAYI BAŞLAT
koku_tipleri = ['Kör', 'Anlık', 'Hafızalı']
suru_tipleri = ['Yok', 'Yan_Lob', 'Ana_Lob']
enerji_tipleri = ['Sabit', 'Yorulan']

all_results = []
print("🌍 EVRENSEL TURNUVA BAŞLIYOR!\n")

for t_name, t_func in terrains.items():
    print(f"\n🌋 Savaş Alanı Yükleniyor: {t_name}")
    
    # PSO
    pso_score = pso_optimize(t_func)
    all_results.append({"Arazi": t_name, "Algoritma": "0_PSO_Klasik", "Final_Hata": pso_score})
    print(f"  -> PSO_Klasik: {pso_score:.6f}")
    
    # FlyOpt Modelleri
    for k in koku_tipleri:
        for s in suru_tipleri:
            for e in enerji_tipleri:
                name = f"Fly_{k}_{s}_{e}"
                score = multi_variant_fly(t_func, k, s, e)
                all_results.append({"Arazi": t_name, "Algoritma": name, "Final_Hata": score})
                # print(f"  -> {name}: {score:.6f}") # Ekranı çok doldurmaması için kapalı
    print(f"  -> Sinek modelleri tamamlandı.")

df_results = pd.DataFrame(all_results)
csv_name = 'Evrensel_Turnuva_Sonuclari.csv'
df_results.to_csv(csv_name, index=False)

In [ ]:
# 4. SONUÇLARI ANALİZ ET VE TABLO ÇİZDİR
# Her arazi için en iyi sineği ve PSO'yu karşılaştıracağız.

pivot_df = df_results.pivot(index="Algoritma", columns="Arazi", values="Final_Hata")
pso_row = pivot_df.loc["0_PSO_Klasik"]
fly_df = pivot_df.drop("0_PSO_Klasik")

best_fly_scores = fly_df.min()
best_fly_names = fly_df.idxmin()

summary_df = pd.DataFrame({
    "Arazi": pivot_df.columns,
    "PSO_Skoru": pso_row.values,
    "En_İyi_Sinek_Skoru": best_fly_scores.values,
    "Şampiyon_Sinek_Modeli": best_fly_names.values
})

print("\n🏆 5 FARKLI COĞRAFYANIN ŞAMPİYONLARI 🏆")
display(summary_df)

# HEATMAP GÖRSELLEŞTİRME (Log Skalası)
plt.figure(figsize=(10, 12))
log_pivot = np.log10(pivot_df + 1e-10)
sns.heatmap(log_pivot, cmap="viridis_r", annot=False)
plt.title('Isı Haritası: 19 Algoritma x 5 Arazi (Koyu/Mor = Daha İyi Skore)')
plt.ylabel('Mimariler')
plt.xlabel('Coğrafyalar')
plt.tight_layout()
grafik_name = 'Evrensel_Turnuva_Heatmap.png'
plt.savefig(grafik_name, dpi=300)
plt.show()

print("\n⬇️ Veriler bilgisayarınıza indiriliyor...")
try:
    files.download(csv_name)
    files.download(grafik_name)
except:
    print("Manuel indirebilirsiniz.")